# 06 — Robustness Check: Excluding Nigeria

This notebook runs the pooled baseline and extended models excluding Nigeria, as a robustness check on the main result from notebook 05.

---

## Motivation

The per-country breakdown in notebook 05 shows that the GDELT improvement in the five-country pooled model is heavily concentrated in Nigeria. Nigeria accounts for approximately 41 events per week on average — roughly three times higher than the next most active country (Mali and Burkina Faso, ~14 events/week) and twenty times higher than Ghana. Nigeria's scale means it dominates the RMSE metric: a large prediction error in one Nigeria week contributes as much to the pooled RMSE as many errors across all other countries combined.

This raises a legitimate question: is the main finding — that GDELT improves conflict prediction — primarily a Nigeria-specific result, or does it hold for the remaining four Sahel-region countries?

This check answers that question directly by re-running the same models on the four-country sample (Mali, Niger, Burkina Faso, Ghana).

---

## What to look for in the results

- If GDELT still improves prediction without Nigeria, the finding is robust and generalisable across the sample.
- If the improvement disappears, the finding is Nigeria-specific and should be interpreted more cautiously.
- Comparing the per-country results without Nigeria to the country-level breakdown from notebook 05 also shows whether removing Nigeria from the training data improves predictions for the remaining countries (because the model no longer has to fit Nigeria's very different scale).

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
import sys
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore")

# Paths relative to repo root
REPO_ROOT = Path().resolve().parent  # one level up from notebooks/

PROC_DIR  = REPO_ROOT / "processed"

MAIN_DIR    = REPO_ROOT / "Results" / "extended"

RESULTS_DIR = REPO_ROOT / "Results" / "robustness"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from model_helpers import (
    HORIZONS, TRAIN_END, TEST_START,
    BASELINE_PREDICTORS, GDELT_PREDICTORS, EXTENDED_PREDICTORS,
    fix_float_col, get_split, fit_negbinom, predict_negbinom,
    evaluate, build_coef_table,
)

print(f"Baseline predictors: {BASELINE_PREDICTORS}")
print(f"Extended predictors: {EXTENDED_PREDICTORS}")

## 1. Load Data and Exclude Nigeria

In [ ]:
panel_full = pd.read_csv(PROC_DIR / "merged_panel.csv", parse_dates=["week_start"])

non_numeric = ["COUNTRY", "week_start"]
for col in panel_full.columns:
    if col not in non_numeric:
        panel_full[col] = fix_float_col(panel_full[col])

# Exclude Nigeria
panel_no_nga = panel_full[panel_full["COUNTRY"] != "Nigeria"].copy()

print(f"Full panel: {panel_full.shape}")
print(f"Without Nigeria: {panel_no_nga.shape}")
print(f"Countries: {sorted(panel_no_nga['COUNTRY'].unique())}")

# Show how much Nigeria was dominating the sample
print("\nMean weekly events by country (training period):")
train_panel = panel_full[panel_full["week_start"] <= TRAIN_END]
print(
    train_panel.groupby("COUNTRY")["n_events"]
    .agg(["mean", "std", lambda x: x.var()/x.mean()])
    .rename(columns={"<lambda_0>": "var/mean"})
    .round(2)
)

## 2. Estimate Baseline and Extended Models (Without Nigeria)

In [ ]:
results_no_nga = {}

for model_label, predictors in [("baseline", BASELINE_PREDICTORS),
                                  ("extended", EXTENDED_PREDICTORS)]:
    results_no_nga[model_label] = {}
    print(f"\n{'='*50}")
    print(f"MODEL: {model_label.upper()} (excluding Nigeria)")
    print("="*50)

    for h in HORIZONS:
        X_tr, X_te, y_tr, y_te, preds = get_split(panel_no_nga, h, predictors)
        result = fit_negbinom(X_tr, y_tr)
        yhat_tr = predict_negbinom(result, X_tr)
        yhat_te = predict_negbinom(result, X_te)
        train_m = evaluate(y_tr, yhat_tr)
        test_m  = evaluate(y_te, yhat_te)

        print(f"  h={h}: Train RMSE={train_m['rmse']:.3f}  Test RMSE={test_m['rmse']:.3f}  "
              f"MAE={test_m['mae']:.3f}")

        results_no_nga[model_label][h] = dict(
            result=result, X_test=X_te, y_test=y_te,
            y_pred_test=yhat_te,
            train_metrics=train_m, test_metrics=test_m,
        )

## 3. Comparison: With vs Without Nigeria

This table shows both the main result (all five countries) and the robustness check (excluding Nigeria) side by side, for both models across all horizons.

In [ ]:
# Load main results from notebook 05
main_metrics = pd.read_csv(MAIN_DIR / "all_metrics.csv")

print("ROBUSTNESS CHECK — POOLED MODEL COMPARISON")
print("=" * 80)
print(f"{'':30} {'h=1':>10} {'h=2':>10} {'h=4':>10}")
print("-" * 80)

rows = []
for model_label in ["baseline", "extended"]:
    for sample, res_dict in [("All 5 countries", None),
                              ("Excl. Nigeria", results_no_nga[model_label])]:
        rmses = []
        for h in HORIZONS:
            if sample == "All 5 countries":
                row = main_metrics[
                    (main_metrics["model"] == model_label) &
                    (main_metrics["horizon"] == h) &
                    (main_metrics["split"] == "test")
                ]
                rmse = row["rmse"].values[0] if len(row) > 0 else np.nan
            else:
                rmse = res_dict[h]["test_metrics"]["rmse"]
            rmses.append(rmse)
            rows.append(dict(model=model_label, sample=sample, horizon=h, rmse=round(rmse,3)))

        label = f"  {model_label.capitalize()} — {sample}"
        print(f"{label:<42} " + "  ".join(f"{r:>8.2f}" for r in rmses))

    # GDELT improvement for this sample
    print()

pd.DataFrame(rows).to_csv(RESULTS_DIR / "robustness_comparison.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'robustness_comparison.csv'}")

# Print GDELT delta for each sample
print("\nGDELT improvement (Extended RMSE - Baseline RMSE):")
for sample_label, model_key in [("All 5 countries", "all5"),
                                  ("Excl. Nigeria", "no_nga")]:
    deltas = []
    for h in HORIZONS:
        if sample_label == "All 5 countries":
            base_r = main_metrics[(main_metrics["model"]=="baseline") &
                                   (main_metrics["horizon"]==h) &
                                   (main_metrics["split"]=="test")]["rmse"].values[0]
            ext_r  = main_metrics[(main_metrics["model"]=="extended") &
                                   (main_metrics["horizon"]==h) &
                                   (main_metrics["split"]=="test")]["rmse"].values[0]
        else:
            base_r = results_no_nga["baseline"][h]["test_metrics"]["rmse"]
            ext_r  = results_no_nga["extended"][h]["test_metrics"]["rmse"]
        deltas.append(ext_r - base_r)
    pct = [100*d/b if b>0 else 0 for d,b in zip(deltas,
           [results_no_nga["baseline"][h]["test_metrics"]["rmse"] for h in HORIZONS]
           if sample_label != "All 5 countries" else
           [main_metrics[(main_metrics["model"]=="baseline")&(main_metrics["horizon"]==h)&
                          (main_metrics["split"]=="test")]["rmse"].values[0] for h in HORIZONS])]
    print(f"  {sample_label}:" + "".join(f"  h={h}: {d:+.2f} ({p:+.1f}%)"
                                          for h,d,p in zip(HORIZONS,deltas,pct)))

## 4. Per-Country Breakdown (Without Nigeria, h = 1)

Shows how the GDELT improvement differs across the four remaining countries when Nigeria is excluded from the pooled model. Without Nigeria contaminating the coefficients, the GDELT signal for Sahel countries may be stronger.

In [ ]:
h = 1

# Get test set for extended model (without Nigeria)
X_test_ext = results_no_nga["extended"][h]["X_test"]
test_meta  = panel_no_nga.loc[X_test_ext.index, ["COUNTRY", "week_start"]].copy()
test_meta["y_true"]      = np.array(results_no_nga["extended"][h]["y_test"])
test_meta["y_pred_ext"]  = results_no_nga["extended"][h]["y_pred_test"]
test_meta["y_pred_base"] = predict_negbinom(
    results_no_nga["baseline"][h]["result"],
    panel_no_nga.loc[X_test_ext.index, BASELINE_PREDICTORS]
)

print("PER-COUNTRY RMSE — Without Nigeria, h=1")
print("=" * 65)
print(f"{'Country':<15} {'Base RMSE':>10} {'Ext RMSE':>10} {'Delta':>8} {'% change':>10}")
print("-" * 65)

country_rows = []
for country in sorted(test_meta["COUNTRY"].unique()):
    sub       = test_meta[test_meta["COUNTRY"] == country]
    base_rmse = np.sqrt(np.mean((sub["y_true"] - sub["y_pred_base"]) ** 2))
    ext_rmse  = np.sqrt(np.mean((sub["y_true"] - sub["y_pred_ext"])  ** 2))
    delta     = ext_rmse - base_rmse
    pct       = 100 * delta / base_rmse
    print(f"  {country:<13} {base_rmse:>10.1f} {ext_rmse:>10.1f} {delta:>+8.1f} {pct:>+9.1f}%")
    country_rows.append(dict(country=country, baseline_rmse=round(base_rmse,2),
                              extended_rmse=round(ext_rmse,2), delta=round(delta,2),
                              pct_change=round(pct,1)))

pd.DataFrame(country_rows).to_csv(RESULTS_DIR / "country_breakdown_no_nigeria_h1.csv", index=False)

## 5. Time Series Plot — Without Nigeria (h = 1)

In [ ]:
remaining_countries = sorted(test_meta["COUNTRY"].unique())
fig, axes = plt.subplots(len(remaining_countries), 1,
                          figsize=(14, 3.5 * len(remaining_countries)), sharex=True)

for ax, country in zip(axes, remaining_countries):
    sub = test_meta[test_meta["COUNTRY"] == country].sort_values("week_start")
    ax.fill_between(sub["week_start"], sub["y_true"],
                    alpha=0.35, color="steelblue", label="Observed")
    ax.plot(sub["week_start"], sub["y_pred_base"],
            color="red", lw=1.2, ls="--", alpha=0.7, label="Baseline")
    ax.plot(sub["week_start"], sub["y_pred_ext"],
            color="darkorange", lw=1.4, label="Extended")
    ax.set_title(country, fontsize=10)
    ax.set_ylabel("Events")
    ax.legend(fontsize=8, loc="upper right")

axes[-1].set_xlabel("Week")
plt.suptitle("Robustness Check (No Nigeria) — Observed and Predicted (h=1, test period)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "robustness_timeseries_h1.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. GDELT Coefficients — Without Nigeria (h = 1)

Comparing GDELT coefficients between the five-country and four-country models shows whether Nigeria's presence was distorting the estimated media effects for other countries.

In [ ]:
print("GDELT COEFFICIENTS — Extended model, h=1")
print("(Standardised — magnitudes are directly comparable)")
print("=" * 55)

# Load five-country coefficients from notebook 05
try:
    coef_full = pd.read_csv(MAIN_DIR / "extended_coefs_h1.csv", index_col=0)
    coef_no_nga = build_coef_table(results_no_nga["extended"][1]["result"])

    gdelt_vars = [v for v in GDELT_PREDICTORS if v in coef_full.index]
    print(f"{'Variable':<22} {'All 5 coef':>12} {'No Nigeria coef':>16}")
    print("-" * 55)
    for v in gdelt_vars:
        c_full = coef_full.loc[v, "Coefficient"] if v in coef_full.index else np.nan
        c_no   = coef_no_nga.loc[v, "Coefficient"] if v in coef_no_nga.index else np.nan
        p_no   = coef_no_nga.loc[v, "p-value"] if v in coef_no_nga.index else np.nan
        sig    = ("***" if p_no<0.001 else "**" if p_no<0.01 else
                  "*" if p_no<0.05 else "(ns)") if not np.isnan(p_no) else ""
        print(f"  {v:<20} {c_full:>12.4f} {c_no:>16.4f}  {sig}")
except FileNotFoundError:
    print("Run notebook 05 first to generate extended_coefs_h1.csv")
    coef_no_nga = build_coef_table(results_no_nga["extended"][1]["result"])
    print("\nCoefficients (no Nigeria only):")
    print(coef_no_nga.to_string())

## 7. Summary

This cell prints a concise summary of the robustness check for the thesis write-up.

In [ ]:
print("ROBUSTNESS CHECK SUMMARY")
print("=" * 50)
print()
print("h=1 results:")
try:
    full_base = main_metrics[(main_metrics["model"]=="baseline") &
                              (main_metrics["horizon"]==1) &
                              (main_metrics["split"]=="test")]["rmse"].values[0]
    full_ext  = main_metrics[(main_metrics["model"]=="extended") &
                              (main_metrics["horizon"]==1) &
                              (main_metrics["split"]=="test")]["rmse"].values[0]
    print(f"  All 5 countries:  Baseline={full_base:.2f}  Extended={full_ext:.2f}  "
          f"Δ={full_ext-full_base:+.2f} ({100*(full_ext-full_base)/full_base:+.1f}%)")
except (KeyError, IndexError):
    print("  (Run notebook 05 first for full-sample results)")

no_nga_base = results_no_nga["baseline"][1]["test_metrics"]["rmse"]
no_nga_ext  = results_no_nga["extended"][1]["test_metrics"]["rmse"]
print(f"  Excl. Nigeria:    Baseline={no_nga_base:.2f}  Extended={no_nga_ext:.2f}  "
      f"Δ={no_nga_ext-no_nga_base:+.2f} ({100*(no_nga_ext-no_nga_base)/no_nga_base:+.1f}%)")
print()
print("Interpretation:")
improvement = no_nga_ext < no_nga_base
if improvement:
    print("  GDELT improves prediction even without Nigeria.")
    print("  The main finding is robust and not purely a Nigeria artefact.")
else:
    print("  GDELT improvement disappears without Nigeria.")
    print("  The main finding is largely Nigeria-specific.")
    print("  This should be acknowledged as a limitation in the thesis.")